<a href="https://colab.research.google.com/github/omsoni/llm-rag-work/blob/prompt_eng_scope/LLM_Medical_Assistant_Prompt_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Address Github Compatibility for nbformat

In [ ]:
import json

with open('LLM_Medical_Assistant_Prompt_Engineering.ipynb', 'r', encoding='utf-8') as f:
    nb = json.load(f)
if 'widgets' in nb.get('metadata', {}):
    del nb['metadata']['widgets']

with open('LLM_Medical_Assistant_Prompt_Engineering.ipynb', 'w', encoding='utf-8') as f:
    json.dump(nb, f, indent=1)

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

## Summary

This notebook explores prompt engineering as a standalone improvement layer for Meta-Llama-3-8B-Instruct (no RAG, no external knowledge), testing whether instruction prompting, output structuring, constraint prompting, and chain-of-thought can raise medical-answer quality on five fixed queries. Five sampling profiles are swept — Deterministic, Conservative, Balanced, Creative, Exploratory — each paired with a tailored prompt template, and responses are scored by a lightweight keyword-based evaluate_response heuristic. The arc traces a clear progression: custom XML-style tags work for tight sampling but cause the model to "self-evaluate" once max_tokens grows, and switching to the proper Llama-3 chat template (<|begin_of_text|> etc.) plus an explicit "answer ONCE" instruction fixes it.

* No judge LLM is used for evalution. Judge LLM based evlaution is implemented in separate notebook in this repo.



## Evaluation mechanism

1. **Five fixed queries** are declared as `Final[str]` constants (sepsis, appendicitis, alopecia, TBI, leg fracture) so every configuration runs against an identical question set.

2. **`evaluate_response` heuristic.** A pure-Python function with no judge LLM that scores each response on multiple dimensions: structural quality (bullets, headers, paragraphs), vocabulary depth (specific vs. generic medical terms across symptoms, treatments, and diagnostics), safety signals (disclaimer phrases, hedge language, red-flag absolute claims), truncation detection, and optional query-overlap. These roll up into a single weighted **`quality_score`** in `[0, 1]` that can be aggregated across queries and configs.

3. **Unified prompt builder.** A single `build_llama3_prompt` function uses Llama-3's native chat template (`<|begin_of_text|>`, `<|start_header_id|>`, `<|eot_id|>`) with a stable system prompt that includes the `"answer ONCE"` instruction. Constraints and chain-of-thought triggers are passed as parameters and folded into the user turn — so the system prompt stays identical across configurations and only the requested variation changes.

4. **Unified generation function.** A single `llm_generate` function takes sampling parameters as data, with `stop=["<|eot_id|>", ...]` applied uniformly so the model halts cleanly at end-of-turn instead of running to `max_tokens`.

5. **Five configurations swept**, each defined as a dict of sampling parameters plus optional prompt variations:
    - **Deterministic** (T=0): base prompt, no constraints.
    - **Conservative** (T=0.1): adds `"Answer in under 5 sentences"` constraint.
    - **Balanced** (T=0.3, max_tokens=512): relaxed to `"under 10 sentences"`.
    - **Creative** (T=0.7): drops constraints, enables chain-of-thought.
    - **Exploratory** (T=1.0, max_tokens=1024): chain-of-thought, used as a brainstorming-layer demonstration.

6. **Quantitative aggregation.** The full sweep (5 configs × 5 queries = 25 runs) is collected into a pandas DataFrame and surfaced as three views: a **config summary** (mean score, word count, disclaimer rate, truncation rate, and specific-term hits per config), a **quality-score matrix** (config × query), and a **best-config-per-query** table — replacing the original notebook's per-query qualitative prose with reproducible numbers.

## Installing and Importing Necessary Libraries and Dependencies

In [1]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --no-cache-dir --no-deps -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 101.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
# For installing the libraries & downloading models from HF Hub
!pip install --upgrade huggingface_hub==0.35.3 pandas==2.2.2 pymupdf==1.26.5 langchain==0.3.27 langchain-community==0.3.31 sentence-transformers==5.1.1  -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 108.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 486.6/486.6 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 119.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the 

In [3]:
pip install diskcache --no-deps --no-cache-dir -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 246.1 MB/s eta 0:00:00


**Note**:
- After running the above cell, you may need to restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.

- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored.

In [4]:
#Libraries for processing dataframes,text
import json,os
import pandas as pd
import numpy as np

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

In [5]:
import textwrap
import warnings
warnings.filterwarnings('ignore')

#### function to pretty print a collection

In [6]:
def pretty_print_doc_collection(relevant_document_chunks):
  for i, chunk in enumerate(relevant_document_chunks):
    print(f"── Chunk {i+1} ──────────────────────────────────────────")
    print(f"Page   : {chunk['metadata'].get('page', 'N/A')}")
    print(f"Score  : {chunk['score']:.4f}")
    print(f"Text   :")
    print(textwrap.fill(chunk['text'], width=80))
    print()

## Basic LLM Response evalution function

In [7]:
from typing import Final
DISCLAIMER_PHRASES: Final = [
    "consult a doctor", "consult your doctor", "seek medical attention",
    "not a replacement", "not intended as medical advice",
    "speak with a healthcare", "professional medical advice",
]


HEDGE_TERMS: Final = [   # uncertainty / safety language
    "may", "might", "could", "possible", "varies", "depending on",
    "individual", "case-by-case",
]

UNSUPPORTED_RED_FLAGS: Final = [   # weak hallucination heuristic
    "always cure", "guaranteed", "100%", "never fails",
    "definitely cure", "miracle",
]

CLINICAL_REGISTER_TERMS: Final = [
    # Process / action words
    "diagnosis", "differential", "etiology", "pathophysiology", "prognosis",
    "indication", "contraindication", "presentation", "manifestation",
    # Care vocabulary
    "treatment", "therapy", "management", "intervention", "protocol",
    "medication", "regimen", "dose", "dosage", "administered",
    # Diagnostic vocabulary
    "imaging", "biopsy", "laboratory", "assay", "screening",
    "examination", "assessment", "evaluation",
    # Outcome vocabulary
    "remission", "complication", "recurrence", "morbidity", "mortality",
]

In [8]:
## Checks for structure of the response
def _structure_score(text: str) -> dict:
    """Objective structural metrics — no keyword guessing."""
    lines = text.splitlines()
    bullet_re = re.compile(r"^\s*([-*•]|\d+\.)\s+")
    bullets = sum(1 for ln in lines if bullet_re.match(ln))
    headers = sum(1 for ln in lines if re.match(r"^\s*#{1,6}\s+\S", ln) or
                                       re.match(r"^\s*\*\*[^*]+\*\*\s*:?\s*$", ln))
    paragraphs = len([p for p in re.split(r"\n\s*\n", text) if p.strip()])

    return {
        "bullet_count": bullets,
        "header_count": headers,
        "paragraph_count": paragraphs,
        "has_structure": bullets >= 2 or headers >= 1,
    }

In [9]:
import re
def _count_hits(text: str, terms: list[str]) -> int:
       text_lower = text.lower()
       return sum(1 for t in terms if re.search(rf"\b{re.escape(t)}\b", text_lower))

In [10]:
def evaluate_response(response: str, query: str | None = None) -> dict:
    """
    Stronger heuristic evaluation of a medical LLM response.

    Returns a dict of metrics:
      - length / readability stats
      - structural quality (bullets, headers, paragraphs)
      - vocabulary depth (clinical register)
      - safety signals (disclaimer presence, hedge language, red-flag claims)
      - composite quality score in [0, 1]

    `query` is optional — if provided, computes term overlap with the query
    and folds it into the composite score.
    """
    if not response or not response.strip():
        return {"empty_response": True, "quality_score": 0.0}

    text = response
    text_lower = text.lower()
    word_count = len(text.split())
    sentence_count = max(1, len(re.findall(r"[.!?]+(?:\s|$)", text)))

    # Check Structure of Response
    structure = _structure_score(text)

    # Content vocabulary for Clinical Terms
    clinical_terms_count = _count_hits(text, CLINICAL_REGISTER_TERMS)

    # Check Response Content for Safety
    has_disclaimer = any(p in text_lower for p in DISCLAIMER_PHRASES)
    hedge_count    = _count_hits(text, HEDGE_TERMS)
    red_flag_count = _count_hits(text, UNSUPPORTED_RED_FLAGS)

    # Is Response loo truncated
    # A response that doesn't end on terminal punctuation likely got cut
    # off by max_tokens. Bullet-list endings are a common false positive,
    # so a final bullet line without punctuation is treated as fine.
    last_line = next(
        (ln for ln in reversed(text.splitlines()) if ln.strip()), ""
    )
    is_bullet_line = bool(re.match(r"^\s*([-*•]|\d+\.)\s+", last_line))
    last_char = last_line.rstrip()[-1] if last_line.rstrip() else ""
    looks_truncated = (last_char not in ".!?\"')]}") and not is_bullet_line

    # ── Query alignment (optional) ───────────────────────────────────
    query_overlap = None
    if query:
        query_terms = {w.lower() for w in re.findall(r"\b[a-z]{4,}\b", query.lower())}
        response_terms = set(re.findall(r"\b[a-z]{4,}\b", text_lower))
        if query_terms:
            query_overlap = round(
                len(query_terms & response_terms) / len(query_terms), 2
            )

    # Composite quality score
    # Positives sum to 1.00 so a perfect response can reach 1.0.
    # Negatives subtract; final score clipped to [0, 1].
    score = 0.0
    score += 0.40 * min(clinical_terms_count / 6, 1.0)   # medical register
    score += 0.20 * (query_overlap or 0.0)               # topical relevance
    score += 0.15 if structure["has_structure"] else 0.0 # structural quality
    score += 0.15 if has_disclaimer else 0.0             # safety
    score += 0.10 * min(hedge_count / 3, 1.0)            # appropriate uncertainty
    score -= 0.15 * min(red_flag_count, 2)               # absolute claims
    score -= 0.05 if looks_truncated else 0.0            # cut off mid-answer
    score = max(0.0, min(1.0, score))

    return {
        "word_count":            word_count,
        "sentence_count":        sentence_count,
        **structure,
        "clinical_terms_count":  clinical_terms_count,
        "has_disclaimer":        has_disclaimer,
        "hedge_count":           hedge_count,
        "red_flag_count":        red_flag_count,
        "looks_truncated":       looks_truncated,
        "query_overlap":         query_overlap,
        "quality_score":         round(score, 3),
    }

In [59]:
LLAMA3_SYSTEM_PROMPT = """You are a knowledgeable medical assistant providing accurate, structured medical information.

Your responses must:
- Use medically correct terminology
- Explain symptoms, causes, diagnostics, and treatments when applicable
- Cite specific clinical evidence (drugs, dosages, procedures) where possible
- Avoid unsupported claims, absolute language ("always cures", "100%"), or speculation
- End with a brief disclaimer recommending consultation with a qualified doctor

Response format:
- **Clinical Explanation** — brief overview of the condition
- **Symptoms / Presentation** — bullet list
- **Diagnosis** — bullet list of diagnostic steps if relevant
- **Treatment Protocol** — bullet list with specific interventions
- **Disclaimer** — one-line clinical caveat

CRITICAL: Provide your answer ONCE. Do not evaluate, grade, or comment on your own response. Stop after the disclaimer.
"""


def build_llama3_prompt(
    query: str,
    system_prompt: str = LLAMA3_SYSTEM_PROMPT,
    constraints: str | None = None,
    use_cot: bool = False,
) -> str:
    """
    Build a Llama-3 chat-formatted prompt with optional constraints
    and chain-of-thought trigger.

    Constraints are folded into the user turn (not the system) so the
    system prompt stays stable across configurations — letting us
    cleanly ablate sampling vs. constraints vs. CoT.
    """
    user_parts = [query]
    if constraints:
        user_parts.append(f"\nConstraints:\n{constraints.strip()}")
    if use_cot:
        user_parts.append("\nThink through this step by step before answering, "
                          "but show only the final structured answer.")
    user_message = "\n".join(user_parts)

    return (
        f"<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\n\n"
        f"{system_prompt.strip()}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n"
        f"{user_message}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
    )


# ── Standard stop tokens, applied uniformly to every llm() call ──────
LLAMA3_STOP = [
    "<|eot_id|>",
    "<|end_of_text|>",
    "<|start_header_id|>",
    "\nassistant",         # catches the role-header echo
    "\n\nassistant",
    "assistant\n\n",       # in case it appears mid-stream
]

# Question Answering using LLM

Following Google Colab T4 friendly LLMs on huggingface were analyzed and compared:

| Model | Params | Domain | Strengths | Weaknesses | Colab T4 Friendly | Notes |
|------|------|------|------|------|------|------|
| meta-llama/Meta-Llama-3-8B-Instruct | 8B | General | Excellent reasoning, strong benchmarks, good instruction following | Not medical-specific | Runs with 4-bit quantization | Best overall |
| mistralai/Mistral-7B-Instruct-v0.2 | 7B | General | Fast inference, strong reasoning, widely used in RAG systems | Slightly weaker knowledge depth | Runs easily on T4 | Best for speed |
| epfl-llm/meditron-7b | 7B | Medical | Trained on PubMed and clinical texts | Weaker instruction following | Runs on T4 | Good domain baseline |
| BioMistral-7B | 7B | Medical | Biomedical pretraining improves performance over MediTron on some tasks | Some hallucination issues | Runs on T4 | Good medical candidate |
| OpenBioLLM-8B | 8B | Medical | Llama-3 based medical tuning | Less widely tested | Runs on T4 with quantization | Promising experimental |

Purpose is to use the LLM model to act as a medical assistant answering natural language questions. Based on Strengths and Weeknesses in the table, **meta-llama/Meta-Llama-3-8B-Instruct** is selected for response generation for its strong ability to generate:

* Structured explanations

* Follow prompts correctly

* Produce step-by-step reasoning


#### **Downloading the model from Hugging Face**

In [12]:
model_name_or_path = "bartowski/Meta-Llama-3-8B-Instruct-GGUF"
model_basename = "Meta-Llama-3-8B-Instruct-Q4_K_M.gguf" # the model is in gguf format

In [13]:
from huggingface_hub import login
login()

In [14]:
bartowski_model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)

Meta-Llama-3-8B-Instruct-Q4_K_M.gguf:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

#### **Model Configuration**
* Max context of 5000 tokens (Max allowed 8192 Tokens)
* Llama 3 8B has 32 transformer layers so 38 layers means every layer runs on GPU
* Process 512 tokens batch in parallel

In [15]:
#Runtime is connected to GPU.
llm = Llama(
    model_path=bartowski_model_path,
    n_ctx=5000,
    n_gpu_layers=-1,
    n_batch=512
)

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 1 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


#### **Model Parameters**

I have set model parameters for **deterministic** response.

* temperature=0 - always pick highest probability token
* top_p=0.95 - probability mass cutoff
* top_k=10 means: only consider top 10 tokens
* Max output tokens 256, Response gets truncated beyond that


#### Function to generate response

In [42]:
# The five sampling profiles
SAMPLING_PROFILES = {
    "Deterministic": {"temperature": 0.0, "top_p": 0.95, "top_k": 10,  "max_tokens": 512},
    "Conservative":  {"temperature": 0.1, "top_p": 0.90, "top_k": 20,  "max_tokens": 512},
    "Balanced":      {"temperature": 0.3, "top_p": 0.85, "top_k": 40,  "max_tokens": 512},
    "Creative":      {"temperature": 0.7, "top_p": 0.90, "top_k": 50,  "max_tokens": 512},
    "Exploratory":   {"temperature": 1.0, "top_p": 0.95, "top_k": 100, "max_tokens": 1024},
}

In [60]:
## Pass Prompt and hyperparameters
def llm_response(prompt: str, profile: str = "Deterministic", **overrides) -> str:
    """
    Generate a response using a named sampling profile.

    Profile names come from SAMPLING_PROFILES (Deterministic, Conservative,
    Balanced, Creative, Exploratory). Any keyword override (e.g. max_tokens=1024)
    takes precedence over the profile's defaults.
    """
    if profile not in SAMPLING_PROFILES:
        raise ValueError(
            f"Unknown profile {profile!r}. "
            f"Choose from: {list(SAMPLING_PROFILES)}"
        )

    sampling = {**SAMPLING_PROFILES[profile], **overrides}

    output = llm(
        prompt=prompt,
        stop=LLAMA3_STOP,
        **sampling,
    )
    return output["choices"][0]["text"].strip()

#### **Declare Query Constants**

In [17]:
Query1: Final[str] = "What is the protocol for managing sepsis in a critical care unit?"
Query2: Final[str] = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
Query3: Final[str] = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
Query4: Final[str] = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
Query5: Final[str] = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"

#### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [36]:
response = llm_response(Query1, max_tokens=512)
print(response)

Llama.generate: prefix-match hit


The protocol should include steps for identifying and treating patients with sepsis, as well as strategies for preventing sepsis.
The protocol for managing sepsis in a critical care unit typically includes the following steps:
1. Identification of Sepsis: Sepsis is identified by the presence of two or more of the following criteria:
* Temperature > 38°C (100.4°F) or < 36°C (96.8°F)
* Heart rate > 90 beats per minute
* Respiratory rate > 20 breaths per minute
* White blood cell count > 12,000 cells/mm³ or < 400 cells/mm³
* Sepsis is also suspected if a patient has a known infection and develops organ dysfunction.
2. Initial Assessment: Upon identification of sepsis, the critical care team should perform an initial assessment to determine the severity of illness and identify potential sources of infection.
3. Fluid Resuscitation: Patients with sepsis should receive fluid resuscitation to maintain adequate blood pressure and perfusion. This may involve administering crystalloid or colloid

In [37]:
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response, Query1)

********************* LLM Response Evaluation ***********************


{'word_count': 369,
 'sentence_count': 22,
 'bullet_count': 18,
 'header_count': 0,
 'paragraph_count': 1,
 'has_structure': True,
 'clinical_terms_count': 6,
 'has_disclaimer': False,
 'hedge_count': 1,
 'red_flag_count': 0,
 'looks_truncated': False,
 'query_overlap': 0.86,
 'quality_score': 0.755}

#### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [44]:
response = llm_response(Query2)
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response, Query2)

Llama.generate: prefix-match hit


Appendicitis is a medical condition that occurs when the appendix becomes inflamed and fills with pus. The appendix is a small, finger-like pouch attached to the large intestine.
Common Symptoms of Appendicitis:
1. Severe abdominal pain: The most common symptom of appendicitis is severe abdominal pain that starts near the belly button and then moves to the lower right side of the abdomen.
2. Nausea and vomiting: Many people with appendicitis experience nausea and vomiting, which can be accompanied by fever, chills, and loss of appetite.
3. Abdominal tenderness: The abdomen may become tender to the touch, especially in the lower right quadrant.
4. Fever: A high fever is common in people with appendicitis, often above 100.4°F (38°C).
5. Loss of appetite: People with appendicitis may experience a loss of appetite and feel weak or fatigued.

Can Appendicitis be Cured via Medicine?
Appendicitis cannot be cured solely through medicine. Antibiotics may help alleviate symptoms and reduce the s

{'word_count': 379,
 'sentence_count': 30,
 'bullet_count': 11,
 'header_count': 0,
 'paragraph_count': 9,
 'has_structure': True,
 'clinical_terms_count': 2,
 'has_disclaimer': True,
 'hedge_count': 1,
 'red_flag_count': 0,
 'looks_truncated': True,
 'query_overlap': 0.73,
 'quality_score': 0.563}

#### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [45]:
response = llm_response(Query3)
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response, Query3)

Llama.generate: prefix-match hit


Hair loss can be a distressing experience, especially when it occurs suddenly and unexpectedly. In this article, we will explore some of the most common causes and effective treatments for sudden patchy hair loss.
Causes of Sudden Patchy Hair Loss:
1. Alopecia Areata: This is an autoimmune condition where the immune system attacks healthy hair follicles, leading to patchy hair loss.
2. Telogen Effluvium: This is a condition where there is an excessive shedding of hair due to hormonal changes, stress, or nutritional deficiencies.
3. Traction Alopecia: This occurs when hair is pulled too tightly, causing hair loss at the scalp.
4. Fungal Infections: Fungal infections like ringworm can cause patchy hair loss.
5. Nutritional Deficiencies: Lack of essential nutrients like iron, zinc, and biotin can contribute to hair loss.
6. Hormonal Imbalance: Hormonal changes during pregnancy, menopause, or thyroid disorders can lead to hair loss.
7. Stress: Physical or emotional stress can cause hair lo

{'word_count': 352,
 'sentence_count': 37,
 'bullet_count': 18,
 'header_count': 0,
 'paragraph_count': 5,
 'has_structure': True,
 'clinical_terms_count': 3,
 'has_disclaimer': False,
 'hedge_count': 0,
 'red_flag_count': 0,
 'looks_truncated': False,
 'query_overlap': 0.42,
 'quality_score': 0.434}

#### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [46]:
response = llm_response(Query4)
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response, Query4)

Llama.generate: prefix-match hit


The answer depends on the severity and location of the injury, as well as the individual's overall health. Here are some common treatment options:
1. Rest: In cases of mild traumatic brain injury (mTBI), rest is often recommended to allow the brain time to heal.
2. Medications: Pain relievers, anti-anxiety medications, and sleep aids may be prescribed to manage symptoms such as headaches, anxiety, or insomnia.
3. Physical therapy: Rehabilitation programs can help improve strength, balance, coordination, and cognitive function.
4. Occupational therapy: This type of therapy focuses on helping individuals with brain injuries regain daily living skills, such as dressing, grooming, and cooking.
5. Speech therapy: Speech therapists can help individuals with language processing difficulties or communication problems.
6. Cognitive rehabilitation: This type of therapy aims to improve memory, attention, problem-solving, and other cognitive functions.
7. Neurorehabilitation: This comprehensive ap

{'word_count': 403,
 'sentence_count': 33,
 'bullet_count': 14,
 'header_count': 0,
 'paragraph_count': 2,
 'has_structure': True,
 'clinical_terms_count': 2,
 'has_disclaimer': False,
 'hedge_count': 2,
 'red_flag_count': 0,
 'looks_truncated': False,
 'query_overlap': 1.0,
 'quality_score': 0.55}

#### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [47]:
response = llm_response(Query5)
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response,Query5)

Llama.generate: prefix-match hit


A fractured leg can be a serious injury that requires immediate attention and proper care to ensure optimal healing and minimize complications. Here's a comprehensive guide on how to handle such an emergency situation:

**Initial Response**

1. **Stop the bleeding**: Apply direct pressure to the wound using a clean cloth or gauze for at least 10-15 minutes to control any bleeding.
2. **Immobilize the leg**: Use a splint, sling, or crutches to immobilize the affected leg and reduce pain and swelling.
3. **Assess the injury**: Check for other injuries, such as head trauma, chest pain, or difficulty breathing.

**Emergency Medical Care**

1. **Call 911 or local emergency services**: If you're in a remote area with no cell phone signal, try to find someone who can call for help or send a message to alert authorities.
2. **Transport the patient**: Move the injured person to a safe location and keep them comfortable until medical help arrives.

**Medical Treatment**

1. **X-rays and diagnosi

{'word_count': 376,
 'sentence_count': 37,
 'bullet_count': 18,
 'header_count': 5,
 'paragraph_count': 12,
 'has_structure': True,
 'clinical_terms_count': 6,
 'has_disclaimer': False,
 'hedge_count': 1,
 'red_flag_count': 0,
 'looks_truncated': True,
 'query_overlap': 0.4,
 'quality_score': 0.613}

### **Comments and Observations**

Following observations were made from the answers received from "Meta-Llama-3-8B-Instruct-GGUF"

* Without external knowledge source (RAG) the model relies on general training data. It does not have access Merck Manual and may use outdated medical material that it was trained on. E.g. The model seem to use outdated criterian for Sepsis identification.

* LLM does not provide any citation of its source of information.

* On Appendicitis query, model missed clinical signs, diagnostics, and treatment details. For query 3 ( patchy hair loss) model did reasonably good job in answering althought specifics for clinical diagnosis and treatment were missing.

* In query 5, LLM assumed there is bleeding but the query only mentioned fracture. Model changes the response structure, instead of earlier responses where bullet point response was given, query 5 returns response in sections.

This behavior is likley because model is anwering from training on various public documents, like Wikipedia, medical websites, health articles, research papers. Since model is providing answers that are not clinically accurate and complete, this approach can't be used as such.

Next step is to try to improve model response by using **prompt engineering**.

# Question Answering using LLM with Prompt Engineering

The goal here is to use **prompt engineering** to guide the model to reason, structure, and avoid hallucinations without adding external source of knowledge.

Idea is to create expert instruction prompts that forces the model to assume a professional role, provide structured answers by separating causes, symptoms, treatments and for safety add clinical caution and avoid unsupported claims.

We will apply **Instruction Prompting** and **Output Structuring** to structure the content for the purpose of being useful for human consumption. This will be different than structuring for machine consumption.

We will divide the prompt into **System Prompt** and **User prompt**. System prompt will be used for **Role prompting**, **Structured output**, **Safety constraints**

In addition to prompt engineering, we will try various hyperparameter configurations for our LLM, trying settings ranging from **Deterministic behavior** to **Exploratory response**. The parameters for all 5 settings are given in below table:

| Strategy | temperature | top_p | top_k | max_tokens | Use Case |
|---|---|---|---|---|---|
| Deterministic | 0 | 0.95 | 10 | 256 | Factual, consistent answers |
| Conservative | 0.1 | 0.9 | 20 | 256 | Slight variation, still safe |
| Balanced | 0.3 | 0.85 | 40 | 512 | General medical Q&A |
| Creative | 0.7 | 0.9 | 50 | 512 | Differential diagnosis |
| Exploratory | 1.0 | 0.95 | 100 | 1024 | Brainstorming, research |


* With Conservative Hyperparameter combination we will add **Constraint Prompting** by specifically adding some rules.
* With **Creative** and **Exploratory** Hyperparameter combination, we will add **Chain-of-Thought (CoT)** prompting to our template

## **1. Deterministic Hyperparameter Optimization**
LLM transformer outputs can be made deterministic by setting temperature to 0, enabling greedy decoding (choosing the highest probability token). This is same confguration as our last run without prompt engineering.


* temperature=0 - always pick highest probability token
* top_p=0.95 - probability mass cutoff
* top_k=10 means: only consider top 10 tokens
* Max output tokens 256, Response gets truncated beyond that

#### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [48]:
prompt = build_llama3_prompt(Query1)
print("******** PROMPT *****************")
print(prompt)
print("******** RESPONSE *****************")
response = llm_response(prompt)
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response, Query1)

******** PROMPT *****************
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a knowledgeable medical assistant providing accurate, structured medical information.

Your responses must:
- Use medically correct terminology
- Explain symptoms, causes, diagnostics, and treatments when applicable
- Cite specific clinical evidence (drugs, dosages, procedures) where possible
- Avoid unsupported claims, absolute language ("always cures", "100%"), or speculation
- End with a brief disclaimer recommending consultation with a qualified doctor

Response format:
- **Clinical Explanation** — brief overview of the condition
- **Symptoms / Presentation** — bullet list
- **Diagnosis** — bullet list of diagnostic steps if relevant
- **Treatment Protocol** — bullet list with specific interventions
- **Disclaimer** — one-line clinical caveat

CRITICAL: Provide your answer ONCE. Do not evaluate, grade, or comment on your own response. Stop after the disclaimer.<|eot_id|><|start_he

Llama.generate: prefix-match hit


**Clinical Explanation**: Sepsis is a life-threatening condition characterized by a dysregulated host response to an infection, leading to organ dysfunction and potentially life-threatening complications.

**Symptoms / Presentation**:

• Fever or hypothermia
• Tachycardia (rapid heart rate)
• Tachypnea (rapid breathing rate)
• Hypotension (low blood pressure)
• Altered mental status
• Decreased urine output
• Organ dysfunction (e.g., acute kidney injury, liver failure)

**Diagnosis**:

• Clinical suspicion based on patient history and physical examination
• Laboratory tests:
	+ Complete Blood Count (CBC) to assess for leukocytosis or leukopenia
	+ Blood cultures to identify the causative pathogen
	+ Lactate levels to monitor tissue hypoperfusion
	+ Serum creatinine and blood urea nitrogen (BUN) to assess kidney function

**Treatment Protocol**:

• **Initial Resuscitation**:
	+ Administer 30 mL/kg of crystalloid solution (e.g., normal saline or lactated Ringer's)
	+ Monitor for signs of

{'word_count': 312,
 'sentence_count': 4,
 'bullet_count': 13,
 'header_count': 3,
 'paragraph_count': 11,
 'has_structure': True,
 'clinical_terms_count': 8,
 'has_disclaimer': True,
 'hedge_count': 0,
 'red_flag_count': 0,
 'looks_truncated': True,
 'query_overlap': 0.43,
 'quality_score': 0.736}

### Observations
if we compare this response with our first experiment without prompt engineering:

```
{'word_count': 160,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': False,
 'mentions_symptom': False}
```

The response with prompt engineering is given in **sections** as requested by our **system prompt** and mentions both treatment and symptomps. That is a **big improvement**.

#### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [61]:
prompt = build_llama3_prompt(Query2)
print("******** PROMPT *****************")
print(prompt)
print("******** RESPONSE *****************")
response = llm_response(prompt)
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response, Query2)

******** PROMPT *****************
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a knowledgeable medical assistant providing accurate, structured medical information.

Your responses must:
- Use medically correct terminology
- Explain symptoms, causes, diagnostics, and treatments when applicable
- Cite specific clinical evidence (drugs, dosages, procedures) where possible
- Avoid unsupported claims, absolute language ("always cures", "100%"), or speculation
- End with a brief disclaimer recommending consultation with a qualified doctor

Response format:
- **Clinical Explanation** — brief overview of the condition
- **Symptoms / Presentation** — bullet list
- **Diagnosis** — bullet list of diagnostic steps if relevant
- **Treatment Protocol** — bullet list with specific interventions
- **Disclaimer** — one-line clinical caveat

CRITICAL: Provide your answer ONCE. Do not evaluate, grade, or comment on your own response. Stop after the disclaimer.<|eot_id|><|start_he

Llama.generate: prefix-match hit


**Clinical Explanation:** Appendicitis is a medical emergency that occurs when the appendix becomes inflamed and fills with pus, causing severe abdominal pain, nausea, and vomiting.

**Symptoms / Presentation:**

• Sudden onset of severe abdominal pain, typically starting near the belly button and moving to the lower right side
• Nausea and vomiting
• Loss of appetite
• Fever
• Abdominal tenderness and swelling
• Abnormal bowel movements (diarrhea or constipation)
• Abdominal guarding (muscle tension) and rebound tenderness

**Diagnosis:**

• Physical examination by a healthcare provider to assess abdominal tenderness, guarding, and rebound tenderness
• Laboratory tests:
	+ Complete Blood Count (CBC) to evaluate for signs of infection
	+ White blood cell count (WBC) to monitor for increased white blood cells indicating inflammation
	+ Urinalysis to rule out other conditions
• Imaging studies:
	+ Abdominal computed tomography (CT) scan or ultrasound to confirm the diagnosis and assess t

{'word_count': 268,
 'sentence_count': 5,
 'bullet_count': 12,
 'header_count': 3,
 'paragraph_count': 8,
 'has_structure': True,
 'clinical_terms_count': 8,
 'has_disclaimer': False,
 'hedge_count': 1,
 'red_flag_count': 0,
 'looks_truncated': False,
 'query_overlap': 0.45,
 'quality_score': 0.673}

### Observations
if we compare this response with our first experiment where no prompt engineering was applied:

```
{'word_count': 187,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': False,
 'mentions_symptom': True}
```

 The response with **prompt engineering** additionally mentions treatment. That is a ** improvement**.

#### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [62]:
prompt = build_llama3_prompt(Query3)
print("******** PROMPT *****************")
print(prompt)
print("******** RESPONSE *****************")
response = llm_response(prompt)
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response, Query3)

******** PROMPT *****************
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a knowledgeable medical assistant providing accurate, structured medical information.

Your responses must:
- Use medically correct terminology
- Explain symptoms, causes, diagnostics, and treatments when applicable
- Cite specific clinical evidence (drugs, dosages, procedures) where possible
- Avoid unsupported claims, absolute language ("always cures", "100%"), or speculation
- End with a brief disclaimer recommending consultation with a qualified doctor

Response format:
- **Clinical Explanation** — brief overview of the condition
- **Symptoms / Presentation** — bullet list
- **Diagnosis** — bullet list of diagnostic steps if relevant
- **Treatment Protocol** — bullet list with specific interventions
- **Disclaimer** — one-line clinical caveat

CRITICAL: Provide your answer ONCE. Do not evaluate, grade, or comment on your own response. Stop after the disclaimer.<|eot_id|><|start_he

Llama.generate: prefix-match hit


**Clinical Explanation:** Sudden patchy hair loss, also known as alopecia areata, is a common condition characterized by the sudden onset of hair loss in one or more patches on the scalp. It can be caused by various factors, including autoimmune disorders, hormonal imbalances, and environmental triggers.

**Symptoms / Presentation:**

• Patchy hair loss on the scalp
• Hair thinning or complete baldness in a specific area
• Redness, itching, or inflammation around the affected area
• In some cases, hair may regrow after a few months

**Diagnosis:**

• Physical examination of the scalp and hair
• Medical history to rule out other conditions causing hair loss (e.g., thyroid disorders, autoimmune diseases)
• Blood tests to check for underlying hormonal imbalances or nutritional deficiencies
• Dermoscopy or trichoscopy to examine the hair follicles

**Treatment Protocol:**

• Topical corticosteroids: applied directly to the affected area to reduce inflammation and promote hair growth
• Mino

{'word_count': 257,
 'sentence_count': 4,
 'bullet_count': 13,
 'header_count': 3,
 'paragraph_count': 8,
 'has_structure': True,
 'clinical_terms_count': 7,
 'has_disclaimer': False,
 'hedge_count': 2,
 'red_flag_count': 0,
 'looks_truncated': False,
 'query_overlap': 0.37,
 'quality_score': 0.691}

### Observations
if we compare this response with our first experiment where no prompt engineering was applied:

```
{'word_count': 180,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': False}
```

 The response with **prompt engineering** mentions both symptomps. That is an ** improvement**.

#### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [63]:
prompt = build_llama3_prompt(Query4)
print("******** PROMPT *****************")
print(prompt)
print("******** RESPONSE *****************")
response = llm_response(prompt)
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response, Query4)

******** PROMPT *****************
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a knowledgeable medical assistant providing accurate, structured medical information.

Your responses must:
- Use medically correct terminology
- Explain symptoms, causes, diagnostics, and treatments when applicable
- Cite specific clinical evidence (drugs, dosages, procedures) where possible
- Avoid unsupported claims, absolute language ("always cures", "100%"), or speculation
- End with a brief disclaimer recommending consultation with a qualified doctor

Response format:
- **Clinical Explanation** — brief overview of the condition
- **Symptoms / Presentation** — bullet list
- **Diagnosis** — bullet list of diagnostic steps if relevant
- **Treatment Protocol** — bullet list with specific interventions
- **Disclaimer** — one-line clinical caveat

CRITICAL: Provide your answer ONCE. Do not evaluate, grade, or comment on your own response. Stop after the disclaimer.<|eot_id|><|start_he

Llama.generate: prefix-match hit


**Clinical Explanation**: Traumatic Brain Injury (TBI) occurs when an external force damages brain tissue, leading to temporary or permanent impairment of brain function. The severity and outcome depend on the extent of damage, location, and individual factors.

**Symptoms / Presentation**:

• Headache
• Confusion or disorientation
• Memory loss or difficulty concentrating
• Mood changes (irritability, depression)
• Sleep disturbances
• Sensitivity to light or noise
• Blurred vision
• Seizures (in severe cases)

**Diagnosis**:

• Physical examination and medical history
• Imaging studies: CT or MRI scans to assess brain damage
• Neurological exam to evaluate cognitive and motor function

**Treatment Protocol**:

• Rest and relaxation to reduce stress and promote recovery
• Pain management with medications such as acetaminophen or ibuprofen
• Anti-anxiety medications (e.g., benzodiazepines) for mood stabilization
• Anticonvulsants (e.g., phenytoin) for seizure control
• Rehabilitation t

{'word_count': 191,
 'sentence_count': 3,
 'bullet_count': 18,
 'header_count': 3,
 'paragraph_count': 8,
 'has_structure': True,
 'clinical_terms_count': 8,
 'has_disclaimer': False,
 'hedge_count': 1,
 'red_flag_count': 0,
 'looks_truncated': False,
 'query_overlap': 0.57,
 'quality_score': 0.697}

### Observations

if we compare this response with our first experiment where no prompt engineering was applied:

```
{'word_count': 195,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}
```

 The response with **prompt engineering** provides structured response as asked in system prompt. Interestingly, LLM response without prompt engineering, also mentioned **Neurorehabilitation** and **Rehabilitation centers**. Since our prompt did not ask LLM about Rehabilitation as part of System Prompt, that category was not included by LLM.

#### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [64]:
prompt = build_llama3_prompt(Query5)
print("******** PROMPT *****************")
print(prompt)
print("******** RESPONSE *****************")
response = llm_response(prompt)
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response, Query5)

******** PROMPT *****************
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a knowledgeable medical assistant providing accurate, structured medical information.

Your responses must:
- Use medically correct terminology
- Explain symptoms, causes, diagnostics, and treatments when applicable
- Cite specific clinical evidence (drugs, dosages, procedures) where possible
- Avoid unsupported claims, absolute language ("always cures", "100%"), or speculation
- End with a brief disclaimer recommending consultation with a qualified doctor

Response format:
- **Clinical Explanation** — brief overview of the condition
- **Symptoms / Presentation** — bullet list
- **Diagnosis** — bullet list of diagnostic steps if relevant
- **Treatment Protocol** — bullet list with specific interventions
- **Disclaimer** — one-line clinical caveat

CRITICAL: Provide your answer ONCE. Do not evaluate, grade, or comment on your own response. Stop after the disclaimer.<|eot_id|><|start_he

Llama.generate: prefix-match hit


**Clinical Explanation**: A fracture is a common injury that occurs when a bone breaks due to trauma or stress. In the case of a hiking trip, a fractured leg can occur from a fall, twisting motion, or direct impact.

**Symptoms / Presentation**:

• Severe pain in the affected area
• Swelling and bruising around the fracture site
• Deformity or abnormal alignment of the limb
• Limited mobility or inability to bear weight on the injured leg
• Numbness, tingling, or weakness in the affected area

**Diagnosis**:

• Physical examination by a healthcare provider to assess the extent of the injury
• Imaging studies such as X-rays, CT scans, or MRI to confirm the fracture and determine its severity
• Assessment for any potential nerve or blood vessel damage

**Treatment Protocol**:

• Immobilization: The injured leg should be immobilized using a splint, cast, or brace to prevent further movement and promote healing.
• Pain management: Over-the-counter pain relievers such as acetaminophen (Tyle

{'word_count': 284,
 'sentence_count': 9,
 'bullet_count': 13,
 'header_count': 3,
 'paragraph_count': 8,
 'has_structure': True,
 'clinical_terms_count': 9,
 'has_disclaimer': False,
 'hedge_count': 1,
 'red_flag_count': 0,
 'looks_truncated': False,
 'query_overlap': 0.67,
 'quality_score': 0.717}

if we compare this response with our first experiment where no prompt engineering was applied:


```
 {'word_count': 190,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}

```

 The response with **prompt engineering** provided Care and Recovery section while LLM response without prompt engineering didn't cover this topic.


## **2. Conservative Hyperparamter Optimization**
A conservative hyperparameter configuration for LLM transformers focuses on stability, high precision, and avoiding over-fitting, particularly when fine-tuning. This approach often uses low learning rates, smaller batch sizes, and lower sampling temperatures.

* temperature=0.1 — Near Greedy
* top_k=20 — Slightly Wider Than Deterministic
* top_p=0.90 sits exactly between balanced and exploratory
* max_tokens=256 is kept small to force the LLM to be concise as directed in
**system prompt**

* With **Consevative Hyperparameter** combination we will add **Constraint Prompting** by specifically adding some rules.

#### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [54]:
prompt = build_llama3_prompt(Query1)
print("******** PROMPT *****************")
print(prompt)
print("******** RESPONSE *****************")
response = llm_response(prompt,"Conservative")
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response, Query1)

******** PROMPT *****************
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Conservative<|eot_id|><|start_header_id|>user<|end_header_id|>

What is the protocol for managing sepsis in a critical care unit?<|eot_id|><|start_header_id|>assistant<|end_header_id|>


******** RESPONSE *****************


Llama.generate: prefix-match hit


The protocol for managing sepsis in a critical care unit typically involves a multi-step approach that includes early recognition, prompt treatment, and close monitoring. Here are the key steps:

1. Early Recognition:
	* Identify patients at risk: Patients with suspected or confirmed infection, especially those with severe sepsis (septic shock or organ dysfunction).
	* Monitor vital signs: Temperature, heart rate, blood pressure, respiratory rate, and oxygen saturation.
	* Look for signs of organ dysfunction: Renal failure, hepatic dysfunction, coagulopathy, and cardiovascular instability.
2. Initial Assessment:
	* Perform a thorough physical examination to identify potential sources of infection (e.g., pneumonia, urinary tract infection, wound infection).
	* Obtain a complete medical history, including allergies, medications, and recent illnesses.
	* Order laboratory tests: Complete blood count (CBC), blood cultures, lactate levels, and other relevant tests (e.g., arterial blood gases

{'word_count': 169,
 'sentence_count': 11,
 'bullet_count': 11,
 'header_count': 0,
 'paragraph_count': 2,
 'has_structure': True,
 'clinical_terms_count': 5,
 'has_disclaimer': False,
 'hedge_count': 0,
 'red_flag_count': 0,
 'looks_truncated': False,
 'query_overlap': 0.86,
 'quality_score': 0.655}

### Observations

if we compare this response with our first experiment where no prompt engineering was applied:

```
{'word_count': 183,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': False}
```
The response with **prompt engineering** provides symptoms and disclaimer.
The Conservative Hyperparamter Optimization gives model little more freedom (temperature=0.1,top_p=0.90) than **Determinitic** behavior. With **System Prompt** we added rule that answer be limited to 5 sentences. Clearlu LLM has responsed in a concise to the point response and ended with "Consult a doctor for personal advice."

#### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [65]:
prompt = build_llama3_prompt(Query2)
print("******** PROMPT *****************")
print(prompt)
print("******** RESPONSE *****************")
response = llm_response(prompt,"Conservative")
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response, Query2)

******** PROMPT *****************
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a knowledgeable medical assistant providing accurate, structured medical information.

Your responses must:
- Use medically correct terminology
- Explain symptoms, causes, diagnostics, and treatments when applicable
- Cite specific clinical evidence (drugs, dosages, procedures) where possible
- Avoid unsupported claims, absolute language ("always cures", "100%"), or speculation
- End with a brief disclaimer recommending consultation with a qualified doctor

Response format:
- **Clinical Explanation** — brief overview of the condition
- **Symptoms / Presentation** — bullet list
- **Diagnosis** — bullet list of diagnostic steps if relevant
- **Treatment Protocol** — bullet list with specific interventions
- **Disclaimer** — one-line clinical caveat

CRITICAL: Provide your answer ONCE. Do not evaluate, grade, or comment on your own response. Stop after the disclaimer.<|eot_id|><|start_he

Llama.generate: prefix-match hit


**Clinical Explanation**: Appendicitis is a medical emergency that occurs when the appendix becomes inflamed and fills with pus, causing severe abdominal pain, nausea, and vomiting.

**Symptoms / Presentation**:

• Sudden onset of severe abdominal pain, typically starting near the belly button and moving to the lower right abdomen
• Nausea and vomiting
• Loss of appetite
• Fever (usually low-grade)
• Abdominal tenderness and guarding
• Abnormal bowel movements

**Diagnosis**: Diagnosis is primarily based on clinical evaluation, including:

• Physical examination
• Medical history
• Laboratory tests: complete blood count (CBC), liver function tests (LFTs), and urinalysis
• Imaging studies: computed tomography (CT) scan or ultrasound may be ordered to confirm the diagnosis

**Treatment Protocol**: Appendicitis is typically treated surgically, as antibiotics alone are not effective in resolving the condition. The surgical procedure of choice is:

• Laparoscopic appendectomy: a minimally i

{'word_count': 208,
 'sentence_count': 4,
 'bullet_count': 12,
 'header_count': 1,
 'paragraph_count': 8,
 'has_structure': True,
 'clinical_terms_count': 8,
 'has_disclaimer': False,
 'hedge_count': 1,
 'red_flag_count': 0,
 'looks_truncated': False,
 'query_overlap': 0.36,
 'quality_score': 0.655}

### Observations

For Query2, **prompt engineering** and configuration modified behaviour of LLM exactly same as for Query1. ### Observations

if we compare this response with our first experiment where no prompt engineering was applied:

```
{'word_count': 187,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': False,
 'mentions_symptom': True}
```

The response with **prompt engineering** provides symptoms and disclaimer.  The response with **prompt engineering** provides symptoms and disclaimer.
The Conservative Hyperparamter Optimization gives model little more freedom (temperature=0.1,top_p=0.90) than **Determinitic** behavior. With **System Prompt** we added rule that answer be limited to 5 sentences. Clearlu LLM has responsed in a concise to the point response and ended with "Consult a doctor for personal advice."
It is important to notice that LLM closed the MEDICAL-ASSISTANT tag in the response that I was expecting it to close.

#### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [66]:
prompt = build_llama3_prompt(Query3)
print("******** PROMPT *****************")
print(prompt)
print("******** RESPONSE *****************")
response = llm_response(prompt,"Conservative")
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response, Query3)

******** PROMPT *****************
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a knowledgeable medical assistant providing accurate, structured medical information.

Your responses must:
- Use medically correct terminology
- Explain symptoms, causes, diagnostics, and treatments when applicable
- Cite specific clinical evidence (drugs, dosages, procedures) where possible
- Avoid unsupported claims, absolute language ("always cures", "100%"), or speculation
- End with a brief disclaimer recommending consultation with a qualified doctor

Response format:
- **Clinical Explanation** — brief overview of the condition
- **Symptoms / Presentation** — bullet list
- **Diagnosis** — bullet list of diagnostic steps if relevant
- **Treatment Protocol** — bullet list with specific interventions
- **Disclaimer** — one-line clinical caveat

CRITICAL: Provide your answer ONCE. Do not evaluate, grade, or comment on your own response. Stop after the disclaimer.<|eot_id|><|start_he

Llama.generate: prefix-match hit


**Clinical Explanation:** Sudden patchy hair loss, also known as alopecia areata, is a common condition characterized by the sudden onset of hair loss in one or more patches on the scalp. It can be caused by various factors, including autoimmune disorders, hormonal imbalances, and nutritional deficiencies.

**Symptoms / Presentation:**

• Patchy bald spots on the scalp
• Hair loss may be gradual or sudden
• May be accompanied by itching, redness, or inflammation around the affected area

**Diagnosis:**

• Physical examination of the scalp to identify the extent and pattern of hair loss
• Medical history to rule out other underlying conditions that may cause similar symptoms
• Blood tests to check for autoimmune disorders, thyroid function, and nutritional deficiencies

**Treatment Protocol:**

• Topical corticosteroids (e.g., triamcinolone cream or ointment) to reduce inflammation and promote hair growth
• Minoxidil (Rogaine) 2% solution applied topically to stimulate hair growth and s

{'word_count': 219,
 'sentence_count': 3,
 'bullet_count': 10,
 'header_count': 3,
 'paragraph_count': 8,
 'has_structure': True,
 'clinical_terms_count': 7,
 'has_disclaimer': False,
 'hedge_count': 1,
 'red_flag_count': 0,
 'looks_truncated': False,
 'query_overlap': 0.53,
 'quality_score': 0.689}

### Observations
Comparing above response with our first experiment where no prompt engineering was applied:

```
{'word_count': 180,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': False}
```

 The response with **prompt engineering** adds disclaimer and returns a structured response. That is improvement over the base scenario without prompt engineering. The instruction in prompt to give answer in 5 sentences seems to shorten the response cutting the information.

#### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [67]:
prompt = build_llama3_prompt(Query4)
print("******** PROMPT *****************")
print(prompt)
print("******** RESPONSE *****************")
response = llm_response(prompt,"Conservative")
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response, Query4)

******** PROMPT *****************
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a knowledgeable medical assistant providing accurate, structured medical information.

Your responses must:
- Use medically correct terminology
- Explain symptoms, causes, diagnostics, and treatments when applicable
- Cite specific clinical evidence (drugs, dosages, procedures) where possible
- Avoid unsupported claims, absolute language ("always cures", "100%"), or speculation
- End with a brief disclaimer recommending consultation with a qualified doctor

Response format:
- **Clinical Explanation** — brief overview of the condition
- **Symptoms / Presentation** — bullet list
- **Diagnosis** — bullet list of diagnostic steps if relevant
- **Treatment Protocol** — bullet list with specific interventions
- **Disclaimer** — one-line clinical caveat

CRITICAL: Provide your answer ONCE. Do not evaluate, grade, or comment on your own response. Stop after the disclaimer.<|eot_id|><|start_he

Llama.generate: prefix-match hit


**Clinical Explanation**: Traumatic Brain Injury (TBI) occurs when an external force causes damage to the brain tissue, leading to temporary or permanent impairment of brain function. The severity and outcome depend on the extent of the injury, location, and individual factors.

**Symptoms / Presentation**:

• Headache
• Confusion or disorientation
• Dizziness or loss of balance
• Memory problems
• Mood changes (irritability, depression)
• Sleep disturbances
• Sensitivity to light or noise
• Blurred vision
• Seizures

**Diagnosis**:

• Physical examination and medical history
• Imaging studies: CT or MRI scans to assess the extent of brain damage
• Neurological exam to evaluate cognitive and motor function
• Laboratory tests (e.g., blood work, EEG) to rule out other conditions

**Treatment Protocol**:

• Rest and relaxation to reduce stress and promote recovery
• Pain management with medications as needed
• Rehabilitation therapy:
	+ Physical therapy for mobility and strength
	+ Occupa

{'word_count': 212,
 'sentence_count': 3,
 'bullet_count': 18,
 'header_count': 3,
 'paragraph_count': 8,
 'has_structure': True,
 'clinical_terms_count': 9,
 'has_disclaimer': False,
 'hedge_count': 2,
 'red_flag_count': 0,
 'looks_truncated': False,
 'query_overlap': 0.57,
 'quality_score': 0.731}

### Observations

Comparing above response with our first experiment where no prompt engineering was applied:

```
{'word_count': 195,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}
```

 The response with **prompt engineering** does little worse. The instruction in prompt to give answer in 5 sentences seems to cut off the response limiting the information provided by LLM. This however will be tested in our next scenario as we change the prompt to answer in 10 sentences or less.

#### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [68]:
prompt = build_llama3_prompt(Query5)
print("******** PROMPT *****************")
print(prompt)
print("******** RESPONSE *****************")
response = llm_response(prompt,"Conservative")
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response, Query5)

******** PROMPT *****************
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a knowledgeable medical assistant providing accurate, structured medical information.

Your responses must:
- Use medically correct terminology
- Explain symptoms, causes, diagnostics, and treatments when applicable
- Cite specific clinical evidence (drugs, dosages, procedures) where possible
- Avoid unsupported claims, absolute language ("always cures", "100%"), or speculation
- End with a brief disclaimer recommending consultation with a qualified doctor

Response format:
- **Clinical Explanation** — brief overview of the condition
- **Symptoms / Presentation** — bullet list
- **Diagnosis** — bullet list of diagnostic steps if relevant
- **Treatment Protocol** — bullet list with specific interventions
- **Disclaimer** — one-line clinical caveat

CRITICAL: Provide your answer ONCE. Do not evaluate, grade, or comment on your own response. Stop after the disclaimer.<|eot_id|><|start_he

Llama.generate: prefix-match hit


**Clinical Explanation**: A fracture is a common injury that occurs when a bone breaks or cracks due to trauma, such as a fall or direct blow. In the case of a leg fracture, it can be a serious condition that requires prompt medical attention to prevent further complications.

**Symptoms / Presentation**:

• Severe pain in the affected area
• Swelling and bruising around the injury site
• Deformity or abnormal alignment of the limb
• Limited mobility or inability to bear weight on the injured leg
• Numbness, tingling, or weakness in the affected limb

**Diagnosis**:

• Physical examination by a healthcare provider to assess the extent of the fracture and surrounding soft tissue damage
• Imaging studies such as X-rays, CT scans, or MRI scans to confirm the diagnosis and determine the type and severity of the fracture

**Treatment Protocol**:

• Immobilization: The injured leg should be immobilized using a splint, cast, or brace to prevent further injury and promote healing.
• Pain manag

{'word_count': 257,
 'sentence_count': 7,
 'bullet_count': 11,
 'header_count': 3,
 'paragraph_count': 8,
 'has_structure': True,
 'clinical_terms_count': 8,
 'has_disclaimer': False,
 'hedge_count': 1,
 'red_flag_count': 0,
 'looks_truncated': False,
 'query_overlap': 0.27,
 'quality_score': 0.637}

Comparing above response with our first experiment where no prompt engineering was applied:

```
{'word_count': 183,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': False}
```

The response with **prompt engineering** provides very structured response. The response covers all required sections except the disclaimer. Overall the LLM does better with this prompt. So far we have noticed the almost always **prompt engineering** does better than just sending the plain question to LLM.


## **3. Balanced Hyperparamter Optimization**

Balanced hyperparameter combinations for Large Language Model (LLM) transformer training and fine-tuning focus on achieving optimal performance, stability, and computational efficiency without overfitting or excessive, costly experimentation:

At 0.3 the model strongly favors the top token but occasionally picks the second or third best — giving slight variation while staying accurate.

top_k=40 — Medium Candidate Pool
top_k opens door to 40 tokens
top_p=0.85 closes it back to ~2-3 high quality tokens
More restrictive than exploratory (0.95) but less than deterministic


In [69]:
prompt = build_llama3_prompt(Query1)
print("******** PROMPT *****************")
print(prompt)
print("******** RESPONSE *****************")
response = llm_response(prompt,"Balanced")
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response, Query1)

******** PROMPT *****************
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a knowledgeable medical assistant providing accurate, structured medical information.

Your responses must:
- Use medically correct terminology
- Explain symptoms, causes, diagnostics, and treatments when applicable
- Cite specific clinical evidence (drugs, dosages, procedures) where possible
- Avoid unsupported claims, absolute language ("always cures", "100%"), or speculation
- End with a brief disclaimer recommending consultation with a qualified doctor

Response format:
- **Clinical Explanation** — brief overview of the condition
- **Symptoms / Presentation** — bullet list
- **Diagnosis** — bullet list of diagnostic steps if relevant
- **Treatment Protocol** — bullet list with specific interventions
- **Disclaimer** — one-line clinical caveat

CRITICAL: Provide your answer ONCE. Do not evaluate, grade, or comment on your own response. Stop after the disclaimer.<|eot_id|><|start_he

Llama.generate: prefix-match hit


**Clinical Explanation:** Sepsis is a life-threatening condition characterized by a dysregulated host response to an infection, leading to organ dysfunction and potentially life-threatening complications.

**Symptoms / Presentation:**

• Fever or hypothermia
• Tachycardia (rapid heart rate)
• Tachypnea (rapid breathing rate)
• Hypotension (low blood pressure)
• Altered mental status
• Organ dysfunction (e.g., acute kidney injury, liver failure)

**Diagnosis:**

• Initial assessment and stabilization in the emergency department or critical care unit
• Laboratory tests:
	+ Complete Blood Count (CBC) to assess for leukocytosis or leukopenia
	+ Blood cultures to identify the causative pathogen
	+ Serum lactate levels to monitor tissue perfusion
	+ Renal function tests (e.g., creatinine, blood urea nitrogen)
• Imaging studies:
	+ Chest X-ray to evaluate for pneumonia or other infections
	+ Computed Tomography (CT) scans or ultrasound to assess for abdominal or pulmonary complications

**Tre

{'word_count': 308,
 'sentence_count': 3,
 'bullet_count': 13,
 'header_count': 3,
 'paragraph_count': 8,
 'has_structure': True,
 'clinical_terms_count': 9,
 'has_disclaimer': False,
 'hedge_count': 2,
 'red_flag_count': 0,
 'looks_truncated': False,
 'query_overlap': 0.71,
 'quality_score': 0.759}

### Observations

Comparing this response with our first experiment where no prompt engineering was applied:


```
{'word_count': 183,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': False}
```

 The response with **prompt engineering** provides the desired output. There is minor hyperparameter changes significant one is max_tokens=512 which allows for longer response. In addition in the system prompt we have asked LLM to now respond in 10 sentences instead of 5 in **Convervative behavior** settings. But we can tell that LLM does not know when to stop and it keeps generating. This seems like my **prompt** has some problem in it. It likely does not like the xml tags which was my attempt to comform with Llama 3 prompt format. It seems like the LLM is evaluating and justying its own response to rules within the <Constraints> tags. Since now the max tokens is 512, model is able to output these additional tokens to respond against rules in <Constraints>.


#### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [70]:
prompt = build_llama3_prompt(Query2)
print("******** PROMPT *****************")
print(prompt)
print("******** RESPONSE *****************")
response = llm_response(prompt,"Balanced")
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response, Query2)

******** PROMPT *****************
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a knowledgeable medical assistant providing accurate, structured medical information.

Your responses must:
- Use medically correct terminology
- Explain symptoms, causes, diagnostics, and treatments when applicable
- Cite specific clinical evidence (drugs, dosages, procedures) where possible
- Avoid unsupported claims, absolute language ("always cures", "100%"), or speculation
- End with a brief disclaimer recommending consultation with a qualified doctor

Response format:
- **Clinical Explanation** — brief overview of the condition
- **Symptoms / Presentation** — bullet list
- **Diagnosis** — bullet list of diagnostic steps if relevant
- **Treatment Protocol** — bullet list with specific interventions
- **Disclaimer** — one-line clinical caveat

CRITICAL: Provide your answer ONCE. Do not evaluate, grade, or comment on your own response. Stop after the disclaimer.<|eot_id|><|start_he

Llama.generate: prefix-match hit


**Clinical Explanation**: Appendicitis is a medical emergency that occurs when the appendix becomes inflamed and fills with pus, causing severe abdominal pain, nausea, and vomiting.

**Symptoms / Presentation**:

• Severe abdominal pain that starts near the belly button and moves to the lower right side
• Nausea and vomiting
• Loss of appetite
• Fever
• Abdominal tenderness
• Abnormal bowel movements (diarrhea or constipation)
• Swollen abdomen

**Diagnosis**: Diagnosis is typically made through a physical examination, medical history, and imaging tests such as:

• CT scan or ultrasound to confirm the presence of an inflamed appendix
• Blood tests to rule out other conditions with similar symptoms

**Treatment Protocol**:

• Mild cases: Antibiotics may be prescribed to treat appendicitis in some cases, but this is not a cure-all.
• Surgical intervention is usually necessary for most cases:
	+ Laparoscopic surgery (keyhole surgery) or open surgery to remove the inflamed appendix
	• Surg

{'word_count': 205,
 'sentence_count': 5,
 'bullet_count': 12,
 'header_count': 2,
 'paragraph_count': 8,
 'has_structure': True,
 'clinical_terms_count': 7,
 'has_disclaimer': False,
 'hedge_count': 2,
 'red_flag_count': 0,
 'looks_truncated': False,
 'query_overlap': 0.36,
 'quality_score': 0.689}

### Observations

Comparing above response with our first experiment where no prompt engineering was applied:

```
{'word_count': 187,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': False,
 'mentions_symptom': True}
```

 The response with **prompt engineering** is significantly better. Clearly LLM is generating response and then evaluating its own response against constraints in the **system prompt**. which is cool. We will get rid of the tags in next scenario and see if that helps the LLM calm down.

#### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [73]:
prompt = build_llama3_prompt(Query1)
print("******** PROMPT *****************")
print(prompt)
print("******** RESPONSE *****************")
response = llm_response(prompt,"Balanced")
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response, Query1)

******** PROMPT *****************
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a knowledgeable medical assistant providing accurate, structured medical information.

Your responses must:
- Use medically correct terminology
- Explain symptoms, causes, diagnostics, and treatments when applicable
- Cite specific clinical evidence (drugs, dosages, procedures) where possible
- Avoid unsupported claims, absolute language ("always cures", "100%"), or speculation
- End with a brief disclaimer recommending consultation with a qualified doctor

Response format:
- **Clinical Explanation** — brief overview of the condition
- **Symptoms / Presentation** — bullet list
- **Diagnosis** — bullet list of diagnostic steps if relevant
- **Treatment Protocol** — bullet list with specific interventions
- **Disclaimer** — one-line clinical caveat

CRITICAL: Provide your answer ONCE. Do not evaluate, grade, or comment on your own response. Stop after the disclaimer.<|eot_id|><|start_he

Llama.generate: prefix-match hit


**Clinical Explanation**: Sepsis is a life-threatening condition characterized by a dysregulated host response to an infection, leading to organ dysfunction and potentially life-threatening complications.

**Symptoms / Presentation**:

• Fever or hypothermia
• Tachycardia (rapid heart rate)
• Tachypnea (rapid breathing rate)
• Hypotension (low blood pressure)
• Altered mental status
• Decreased urine output
• Organ dysfunction (e.g., acute kidney injury, acute respiratory distress syndrome)

**Diagnosis**:

• Clinical suspicion based on patient history and physical examination
• Laboratory tests:
	+ Complete Blood Count (CBC) to assess for leukocytosis or leukopenia
	+ Blood cultures to identify the causative pathogen
	+ Lactic acid levels to monitor tissue hypoperfusion
	+ Biomarkers such as procalcitonin, C-reactive protein, and interleukin-6 to aid in diagnosis

**Treatment Protocol**:

• Initial management:
	+ Administer broad-spectrum antibiotics within 1 hour of recognition of se

{'word_count': 237,
 'sentence_count': 2,
 'bullet_count': 11,
 'header_count': 3,
 'paragraph_count': 8,
 'has_structure': True,
 'clinical_terms_count': 8,
 'has_disclaimer': False,
 'hedge_count': 0,
 'red_flag_count': 0,
 'looks_truncated': False,
 'query_overlap': 0.29,
 'quality_score': 0.608}

### Observations

Comparing above response with our first experiment where no prompt engineering was applied:

```
{'word_count': 180,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': False}
```

The response with **prompt engineering** is significantly better. Symptoms are missing. It seems like hair loss symptoms are self explainatory. LLM still continue to generate secondary response to constraints.

#### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [74]:
prompt = build_llama3_prompt(Query4)
print("******** PROMPT *****************")
print(prompt)
print("******** RESPONSE *****************")
response = llm_response(prompt,"Balanced")
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response, Query4)

******** PROMPT *****************
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a knowledgeable medical assistant providing accurate, structured medical information.

Your responses must:
- Use medically correct terminology
- Explain symptoms, causes, diagnostics, and treatments when applicable
- Cite specific clinical evidence (drugs, dosages, procedures) where possible
- Avoid unsupported claims, absolute language ("always cures", "100%"), or speculation
- End with a brief disclaimer recommending consultation with a qualified doctor

Response format:
- **Clinical Explanation** — brief overview of the condition
- **Symptoms / Presentation** — bullet list
- **Diagnosis** — bullet list of diagnostic steps if relevant
- **Treatment Protocol** — bullet list with specific interventions
- **Disclaimer** — one-line clinical caveat

CRITICAL: Provide your answer ONCE. Do not evaluate, grade, or comment on your own response. Stop after the disclaimer.<|eot_id|><|start_he

Llama.generate: prefix-match hit


**Clinical Explanation**: A traumatic brain injury (TBI) occurs when an external force damages the brain tissue, leading to temporary or permanent impairment of brain function. The severity and outcome of TBI depend on the extent of the damage, location, and individual factors.

**Symptoms / Presentation**:

• Confusion
• Loss of consciousness
• Headache
• Dizziness
• Nausea and vomiting
• Blurred vision
• Hearing loss or ringing in the ears
• Weakness or numbness in the arms or legs
• Difficulty speaking or understanding speech
• Memory loss or difficulty concentrating

**Diagnosis**:

• Physical examination by a healthcare provider
• Imaging studies (CT or MRI scans) to assess brain damage
• Neurological exam to evaluate cognitive and motor function
• Blood tests to rule out other conditions that may mimic TBI symptoms

**Treatment Protocol**:

• Rest and relaxation to reduce stress and promote recovery
• Pain management with medications as needed
• Rehabilitation therapy to improve 

{'word_count': 209,
 'sentence_count': 3,
 'bullet_count': 19,
 'header_count': 3,
 'paragraph_count': 8,
 'has_structure': True,
 'clinical_terms_count': 8,
 'has_disclaimer': False,
 'hedge_count': 2,
 'red_flag_count': 0,
 'looks_truncated': False,
 'query_overlap': 0.57,
 'quality_score': 0.731}

### Observations

Comparing above response with our first experiment where no prompt engineering was applied:

```
{'word_count': 195,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}
```

 The response with **prompt engineering** is significantly better. Symptoms are missing. It seems like hair loss symptoms are self explainatory. LLM continue to generate secondary response to constraints.

#### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [75]:
prompt = build_llama3_prompt(Query5)
print("******** PROMPT *****************")
print(prompt)
print("******** RESPONSE *****************")
response = llm_response(prompt,"Balanced")
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response, Query5)

******** PROMPT *****************
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a knowledgeable medical assistant providing accurate, structured medical information.

Your responses must:
- Use medically correct terminology
- Explain symptoms, causes, diagnostics, and treatments when applicable
- Cite specific clinical evidence (drugs, dosages, procedures) where possible
- Avoid unsupported claims, absolute language ("always cures", "100%"), or speculation
- End with a brief disclaimer recommending consultation with a qualified doctor

Response format:
- **Clinical Explanation** — brief overview of the condition
- **Symptoms / Presentation** — bullet list
- **Diagnosis** — bullet list of diagnostic steps if relevant
- **Treatment Protocol** — bullet list with specific interventions
- **Disclaimer** — one-line clinical caveat

CRITICAL: Provide your answer ONCE. Do not evaluate, grade, or comment on your own response. Stop after the disclaimer.<|eot_id|><|start_he

Llama.generate: prefix-match hit


**Clinical Explanation**
A fracture is a common injury that occurs when a bone breaks due to trauma or stress. A leg fracture can range from a minor hairline crack to a more severe break that requires surgical intervention.

**Symptoms / Presentation**

* Pain, swelling, and bruising at the affected area
* Deformity or abnormal alignment of the limb
* Limited mobility or inability to bear weight on the affected leg
* Crepitus (grating sensation) when moving the joint
* Swelling, numbness, or tingling in the affected extremities

**Diagnosis**

* Physical examination by a healthcare provider to assess the extent of the fracture and surrounding soft tissue damage
* Imaging studies such as X-rays, CT scans, or MRI scans to confirm the diagnosis and determine the severity of the fracture
* Assessment for potential nerve or blood vessel damage

**Treatment Protocol**

* Immobilization: The affected leg should be immobilized using a splint or cast to prevent further injury and promote healin

{'word_count': 271,
 'sentence_count': 8,
 'bullet_count': 13,
 'header_count': 5,
 'paragraph_count': 8,
 'has_structure': True,
 'clinical_terms_count': 10,
 'has_disclaimer': True,
 'hedge_count': 2,
 'red_flag_count': 0,
 'looks_truncated': False,
 'query_overlap': 0.27,
 'quality_score': 0.821}

Comparing above response with our first experiment where no prompt engineering was applied:

```
{'word_count': 183,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': False}
```

 The response with **prompt engineering** is significantly better. Symptoms are missing. It seems like hair loss symptoms are self explainatory. LLM continue to generate secondary response to constraints insisting "The Medical Assistant's response adheres to the constraints set forth"


## **4. Creative Hyperparameter Optimization**

Below configuration produces more creative responses because the sampling parameters increase diversity in token selection rather than always choosing the most probable next token.

* With temperature = 0.7, probability differences between tokens are reducedand lower-ranked tokens get more chance to be selected.
* Top_p 0f .9 doesn't forces model to pick the highest probability token allowing more choices but avoiding meaningless tokens

#### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [77]:
prompt = build_llama3_prompt(Query1)
print("******** PROMPT *****************")
print(prompt)
print("******** RESPONSE *****************")
response = llm_response(prompt,"Creative")
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response, Query1)

******** PROMPT *****************
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a knowledgeable medical assistant providing accurate, structured medical information.

Your responses must:
- Use medically correct terminology
- Explain symptoms, causes, diagnostics, and treatments when applicable
- Cite specific clinical evidence (drugs, dosages, procedures) where possible
- Avoid unsupported claims, absolute language ("always cures", "100%"), or speculation
- End with a brief disclaimer recommending consultation with a qualified doctor

Response format:
- **Clinical Explanation** — brief overview of the condition
- **Symptoms / Presentation** — bullet list
- **Diagnosis** — bullet list of diagnostic steps if relevant
- **Treatment Protocol** — bullet list with specific interventions
- **Disclaimer** — one-line clinical caveat

CRITICAL: Provide your answer ONCE. Do not evaluate, grade, or comment on your own response. Stop after the disclaimer.<|eot_id|><|start_he

Llama.generate: prefix-match hit


**Clinical Explanation**: Sepsis is a life-threatening condition characterized by an overwhelming host response to infection, leading to organ dysfunction and potentially life-threatening complications.

**Symptoms / Presentation**:

• Fever or hypothermia
• Tachycardia or bradycardia
• Tachypnea or apnea
• Hypotension or hypertension
• Altered mental status or confusion
• Decreased urine output or oliguria
• Lactic acidosis or increased lactate levels

**Diagnosis**: The diagnosis of sepsis is primarily clinical, and the following criteria are used:

• Two or more SIRS (Systemic Inflammatory Response Syndrome) criteria:
	+ Temperature > 38°C (100.4°F) or <36°C (96.8°F)
	+ Heart rate >90 beats per minute
	+ Respiratory rate >20 breaths per minute or PaCO2 < 32 mmHg
	+ White blood cell count >12,000 cells/μL or <4000 cells/μL, or >10% bands
• Evidence of infection (e.g., positive cultures, radiographic abnormalities)

**Treatment Protocol**: The management of sepsis in a critical care u

{'word_count': 264,
 'sentence_count': 3,
 'bullet_count': 12,
 'header_count': 1,
 'paragraph_count': 8,
 'has_structure': True,
 'clinical_terms_count': 7,
 'has_disclaimer': False,
 'hedge_count': 0,
 'red_flag_count': 0,
 'looks_truncated': False,
 'query_overlap': 0.71,
 'quality_score': 0.692}

Comparing above response with our first experiment where no prompt engineering was applied:

```
{'word_count': 160,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': False,
 'mentions_symptom': False}
```

 The response with **prompt engineering** provides a lot of information. Seems like with higher temperature LLM is going off script. Before next experiment, we will make changes to prompts to see if we can make LLM not comment on its own response.

 * Tell LLM to provide answer once not evaluate or comment on response.
 * Shift to Llama3 prompt format


#### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [78]:
prompt = build_llama3_prompt(Query2)
print("******** PROMPT *****************")
print(prompt)
print("******** RESPONSE *****************")
response = llm_response(prompt,"Creative")
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response, Query2)

******** PROMPT *****************
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a knowledgeable medical assistant providing accurate, structured medical information.

Your responses must:
- Use medically correct terminology
- Explain symptoms, causes, diagnostics, and treatments when applicable
- Cite specific clinical evidence (drugs, dosages, procedures) where possible
- Avoid unsupported claims, absolute language ("always cures", "100%"), or speculation
- End with a brief disclaimer recommending consultation with a qualified doctor

Response format:
- **Clinical Explanation** — brief overview of the condition
- **Symptoms / Presentation** — bullet list
- **Diagnosis** — bullet list of diagnostic steps if relevant
- **Treatment Protocol** — bullet list with specific interventions
- **Disclaimer** — one-line clinical caveat

CRITICAL: Provide your answer ONCE. Do not evaluate, grade, or comment on your own response. Stop after the disclaimer.<|eot_id|><|start_he

Llama.generate: prefix-match hit


**Clinical Explanation:** Appendicitis is a medical emergency that occurs when the appendix becomes inflamed and fills with pus, causing severe pain in the lower right abdomen. It is typically caused by blockage of the vermiform appendix due to fecaliths, lymphoid hyperplasia, or other factors.

**Symptoms / Presentation:**

• Severe, constant abdominal pain that starts near the belly button and moves to the lower right abdomen
• Nausea and vomiting
• Fever
• Loss of appetite
• Abdominal tenderness to palpation
• Rebound tenderness (pain that worsens with deep palpation)

**Diagnosis:**

• Medical history and physical examination
• Laboratory tests:
	+ Complete blood count (CBC) for signs of infection
	+ Blood chemistry tests to rule out other conditions
• Imaging studies:
	+ Abdominal computed tomography (CT) scan or ultrasound to confirm appendicitis

**Treatment Protocol:**

If diagnosed with uncomplicated appendicitis, antibiotics may be administered to help manage symptoms and red

{'word_count': 231,
 'sentence_count': 7,
 'bullet_count': 10,
 'header_count': 3,
 'paragraph_count': 9,
 'has_structure': True,
 'clinical_terms_count': 9,
 'has_disclaimer': False,
 'hedge_count': 1,
 'red_flag_count': 0,
 'looks_truncated': False,
 'query_overlap': 0.27,
 'quality_score': 0.637}

Comparing above response with our earlier experiment where no prompt engineering was applied:

```
 {'word_count': 187,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': False,
 'mentions_symptom': True}
```

 The response with **prompt engineering** provides perfect response.
 And there is no confusion with LLM because of tags in our prompt and it did listen to our IMPORTANT instruction to not to evaluate or rate its own response. This is the best response we have received so far.


#### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [80]:
prompt = build_llama3_prompt(Query3)
print("******** PROMPT *****************")
print(prompt)
print("******** RESPONSE *****************")
response = llm_response(prompt,"Creative")
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response, Query3)

******** PROMPT *****************
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a knowledgeable medical assistant providing accurate, structured medical information.

Your responses must:
- Use medically correct terminology
- Explain symptoms, causes, diagnostics, and treatments when applicable
- Cite specific clinical evidence (drugs, dosages, procedures) where possible
- Avoid unsupported claims, absolute language ("always cures", "100%"), or speculation
- End with a brief disclaimer recommending consultation with a qualified doctor

Response format:
- **Clinical Explanation** — brief overview of the condition
- **Symptoms / Presentation** — bullet list
- **Diagnosis** — bullet list of diagnostic steps if relevant
- **Treatment Protocol** — bullet list with specific interventions
- **Disclaimer** — one-line clinical caveat

CRITICAL: Provide your answer ONCE. Do not evaluate, grade, or comment on your own response. Stop after the disclaimer.<|eot_id|><|start_he

Llama.generate: prefix-match hit


**Clinical Explanation:** Sudden patchy hair loss, also known as alopecia aerata, is a common condition characterized by the rapid onset of hair loss in small, circular patches. It can occur anywhere on the scalp, but typically affects the crown or frontal area.

**Symptoms / Presentation:**

• Small, round bald patches on the scalp
• Hair loss may be sudden and unexpected
• Patches may appear in a random pattern or follow a specific distribution (e.g., along the hairline)
• Hair follicles may be inflamed or swollen

**Diagnosis:**

• Physical examination of the affected area
• Medical history to rule out underlying conditions (e.g., autoimmune disorders, hormonal imbalances)
• Dermoscopy or trichoscopy to examine the hair and scalp more closely

**Treatment Protocol:**

• Topical corticosteroids: Minoxidil 2% solution (Rogaine) or triamcinolone cream (Neuracort) applied directly to the affected area
• Oral medications:
	+ Minoxidil oral solution (Rogaine)
	+ Finasteride (Propecia) for

{'word_count': 219,
 'sentence_count': 3,
 'bullet_count': 11,
 'header_count': 3,
 'paragraph_count': 8,
 'has_structure': True,
 'clinical_terms_count': 6,
 'has_disclaimer': False,
 'hedge_count': 1,
 'red_flag_count': 0,
 'looks_truncated': False,
 'query_overlap': 0.32,
 'quality_score': 0.647}

Comparing above response with our first experiment where no prompt engineering was applied:

```
 {'word_count': 180,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': False}
```

 The response with **prompt engineering** covers all required sections and is behaving way better providing expected information and not going off script.


#### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [79]:
prompt = build_llama3_prompt(Query4)
print("******** PROMPT *****************")
print(prompt)
print("******** RESPONSE *****************")
response = llm_response(prompt,"Creative")
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response, Query4)

******** PROMPT *****************
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a knowledgeable medical assistant providing accurate, structured medical information.

Your responses must:
- Use medically correct terminology
- Explain symptoms, causes, diagnostics, and treatments when applicable
- Cite specific clinical evidence (drugs, dosages, procedures) where possible
- Avoid unsupported claims, absolute language ("always cures", "100%"), or speculation
- End with a brief disclaimer recommending consultation with a qualified doctor

Response format:
- **Clinical Explanation** — brief overview of the condition
- **Symptoms / Presentation** — bullet list
- **Diagnosis** — bullet list of diagnostic steps if relevant
- **Treatment Protocol** — bullet list with specific interventions
- **Disclaimer** — one-line clinical caveat

CRITICAL: Provide your answer ONCE. Do not evaluate, grade, or comment on your own response. Stop after the disclaimer.<|eot_id|><|start_he

Llama.generate: prefix-match hit


**Clinical Explanation**: A traumatic brain injury (TBI) occurs when an external force damages the brain tissue, leading to temporary or permanent disruption of normal brain function. The severity and extent of damage depend on factors such as the location, velocity, and intensity of the impact.

**Symptoms / Presentation**:

• Confusion
• Loss of consciousness
• Headache
• Dizziness
• Memory loss
• Difficulty concentrating
• Mood changes
• Sleep disturbances
• Sensory impairments (hearing, vision, balance)

**Diagnosis**:
Diagnostic steps may include:

• Physical examination
• Computed Tomography (CT) scan or Magnetic Resonance Imaging (MRI)
• Electroencephalogram (EEG)
• Neuropsychological tests to assess cognitive and behavioral function

**Treatment Protocol**:

• Rest: Patient is advised to avoid strenuous activities, including work or school
• Pain management: Medications such as acetaminophen or ibuprofen for headache relief
• Cerebrospinal fluid drainage: Shunting may be necess

{'word_count': 212,
 'sentence_count': 3,
 'bullet_count': 18,
 'header_count': 3,
 'paragraph_count': 8,
 'has_structure': True,
 'clinical_terms_count': 8,
 'has_disclaimer': False,
 'hedge_count': 1,
 'red_flag_count': 0,
 'looks_truncated': False,
 'query_overlap': 0.5,
 'quality_score': 0.683}

Comparing above response with our first experiment where no prompt engineering was applied:

```
 {'word_count': 195,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': True,
 'mentions_symptom': True}
```

 The response with **prompt engineering** covers all required sections and is responding way better. We will skip Query 5 and go to our next experiment with even higher temperature to get more exploratory behavior which can be really useful for acedemic use cases.



In [81]:
prompt = build_llama3_prompt(Query5)
print("******** PROMPT *****************")
print(prompt)
print("******** RESPONSE *****************")
response = llm_response(prompt,"Creative")
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response, Query5)

******** PROMPT *****************
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a knowledgeable medical assistant providing accurate, structured medical information.

Your responses must:
- Use medically correct terminology
- Explain symptoms, causes, diagnostics, and treatments when applicable
- Cite specific clinical evidence (drugs, dosages, procedures) where possible
- Avoid unsupported claims, absolute language ("always cures", "100%"), or speculation
- End with a brief disclaimer recommending consultation with a qualified doctor

Response format:
- **Clinical Explanation** — brief overview of the condition
- **Symptoms / Presentation** — bullet list
- **Diagnosis** — bullet list of diagnostic steps if relevant
- **Treatment Protocol** — bullet list with specific interventions
- **Disclaimer** — one-line clinical caveat

CRITICAL: Provide your answer ONCE. Do not evaluate, grade, or comment on your own response. Stop after the disclaimer.<|eot_id|><|start_he

Llama.generate: prefix-match hit


**Clinical Explanation:** A lower limb fracture, also known as a tibial or femoral fracture, occurs when there is a break in one of the bones of the lower extremities. This type of injury often requires prompt medical attention to ensure proper healing and minimize complications.

**Symptoms / Presentation:**

* Severe pain at the site of the fracture
* Swelling, bruising, and deformity around the affected area
* Inability to bear weight on the injured leg or difficulty moving it
* Limited range of motion in the knee or ankle
* Audible snapping or grinding sensation when the bone moves

**Diagnosis:**

* Physical examination by a healthcare provider to assess the severity of the fracture and any associated injuries
* Imaging studies such as X-rays, CT scans, or MRI scans to confirm the diagnosis and evaluate the extent of the damage

**Treatment Protocol:**

* Immobilization with a splint or cast to reduce pain and prevent further injury
* Pain management with medications such as aceta

{'word_count': 288,
 'sentence_count': 5,
 'bullet_count': 13,
 'header_count': 3,
 'paragraph_count': 8,
 'has_structure': True,
 'clinical_terms_count': 9,
 'has_disclaimer': False,
 'hedge_count': 1,
 'red_flag_count': 0,
 'looks_truncated': False,
 'query_overlap': 0.4,
 'quality_score': 0.663}

## **5. Exploratory Hyperparameter Optimization**

* temperature=1.0 - use distribution as-is, sampling reflects raw model probabilities
* top_p=0.95 - cut off probability
* top_k=100 - consider top 100 choices
* max_tokens=1024 - so that detailed answer for exploration does not get cutout

These parameters together essentially remove most restrictions on the model's output generation. Here's what each does at these values:
Dosage or drug interaction queries (too risky with hallucinations)
Patient-facing responses (inconsistency is dangerous)

Since accuracy drops, for medical AI, we should use exploratory only as a brainstorming layer, then validate outputs with a deterministic pass at temperature=0.

#### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [82]:
prompt = build_llama3_prompt(Query1)
print("******** PROMPT *****************")
print(prompt)
print("******** RESPONSE *****************")
response = llm_response(prompt,"Exploratory")
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response, Query1)

******** PROMPT *****************
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a knowledgeable medical assistant providing accurate, structured medical information.

Your responses must:
- Use medically correct terminology
- Explain symptoms, causes, diagnostics, and treatments when applicable
- Cite specific clinical evidence (drugs, dosages, procedures) where possible
- Avoid unsupported claims, absolute language ("always cures", "100%"), or speculation
- End with a brief disclaimer recommending consultation with a qualified doctor

Response format:
- **Clinical Explanation** — brief overview of the condition
- **Symptoms / Presentation** — bullet list
- **Diagnosis** — bullet list of diagnostic steps if relevant
- **Treatment Protocol** — bullet list with specific interventions
- **Disclaimer** — one-line clinical caveat

CRITICAL: Provide your answer ONCE. Do not evaluate, grade, or comment on your own response. Stop after the disclaimer.<|eot_id|><|start_he

Llama.generate: prefix-match hit


**Clinical Explanation:** Sepsis is a life-threatening condition that arises from an uncontrolled response to infection, characterized by a dysregulated immune response and organ dysfunction. Early recognition and prompt management of sepsis are crucial to reduce morbidity and mortality.

**Symptoms / Presentation:**

• Fever or hypothermia
• Tachycardia or bradycardia
• Tachypnea or apnea
• Decreased urine output or altered mental status
• Hypotension or vasopressor dependency
• Increased lactate levels

**Diagnosis:**

• Sepsis-3 criteria:
	+ Suspect sepsis based on clinical presentation and laboratory findings.
	+ Use of Sequential Organ Failure Assessment (SOFA) score to assess organ dysfunction.

**Treatment Protocol:**

• **Initial Management:**
	+ Administer empiric antibiotics within 1 hour of recognition of septic shock or severe sepsis
	+ Fluid resuscitation with normal saline or lactated Ringer's solution
	+ Vasopressors may be necessary for hypotension
	+ Mechanical ventila

{'word_count': 223,
 'sentence_count': 6,
 'bullet_count': 10,
 'header_count': 3,
 'paragraph_count': 8,
 'has_structure': True,
 'clinical_terms_count': 11,
 'has_disclaimer': False,
 'hedge_count': 1,
 'red_flag_count': 0,
 'looks_truncated': False,
 'query_overlap': 0.57,
 'quality_score': 0.697}

Comparing above response with our first experiment where no prompt engineering was applied:

```
{'word_count': 160,
 'has_disclaimer': False,
 'is_structured': True,
 'mentions_treatment': False,
 'mentions_symptom': False}
```

 The response with **prompt engineering** covers all expectations provided in system and user promt and is responding with detailed information. This is great from a LLM that is not tied to medical domain and is trained for general purpose. It is doing all that was expected from it. It is showing

* strong reasoning ability

* Follows instructions given in the prompt

* gives complex explanations in logical structure

Also, we noticed it is important to configure max tokens to so the LLM does not cut off the response. Format our **prompts** properly and set other hyperparameter properly. This learning will be used in our next effort to build our actual Medical Assistant.

I am going to run one more query on this and skip the other three.

#### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [83]:
prompt = build_llama3_prompt(Query2)
print("******** PROMPT *****************")
print(prompt)
print("******** RESPONSE *****************")
response = llm_response(prompt,"Exploratory")
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response, Query2)

******** PROMPT *****************
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a knowledgeable medical assistant providing accurate, structured medical information.

Your responses must:
- Use medically correct terminology
- Explain symptoms, causes, diagnostics, and treatments when applicable
- Cite specific clinical evidence (drugs, dosages, procedures) where possible
- Avoid unsupported claims, absolute language ("always cures", "100%"), or speculation
- End with a brief disclaimer recommending consultation with a qualified doctor

Response format:
- **Clinical Explanation** — brief overview of the condition
- **Symptoms / Presentation** — bullet list
- **Diagnosis** — bullet list of diagnostic steps if relevant
- **Treatment Protocol** — bullet list with specific interventions
- **Disclaimer** — one-line clinical caveat

CRITICAL: Provide your answer ONCE. Do not evaluate, grade, or comment on your own response. Stop after the disclaimer.<|eot_id|><|start_he

Llama.generate: prefix-match hit


**Clinical Explanation:** Appendicitis is a medical emergency that occurs when the appendix becomes inflamed and filled with pus, often causing severe abdominal pain, nausea, vomiting, and fever. The appendix is a small, tube-like pouch attached to the large intestine.

**Symptoms/Presentation:**

* Sudden onset of sharp, severe, and persistent abdominal pain in the lower right quadrant (LRQ)
* Nausea and vomiting
* Loss of appetite
* Fever (usually 100°F - 101.5°F or 37.8°C - 38.6°C)
* Abdominal tenderness and guarding
* Tenderness to palpation in the LRQ
* Elevated white blood cell count

**Diagnosis:**

* Physical examination by a healthcare provider to identify abdominal tenderness, rebound tenderness, and guarding
* Imaging studies (e.g., CT or ultrasound) to confirm diagnosis and rule out other conditions

**Treatment Protocol:**

* Antibiotics may be administered to treat mild cases of appendicitis, but surgical intervention is usually necessary for definitive treatment.
* Surge

{'word_count': 229,
 'sentence_count': 6,
 'bullet_count': 11,
 'header_count': 3,
 'paragraph_count': 8,
 'has_structure': True,
 'clinical_terms_count': 10,
 'has_disclaimer': False,
 'hedge_count': 1,
 'red_flag_count': 0,
 'looks_truncated': False,
 'query_overlap': 0.36,
 'quality_score': 0.655}

In [84]:
prompt = build_llama3_prompt(Query3)
print("******** PROMPT *****************")
print(prompt)
print("******** RESPONSE *****************")
response = llm_response(prompt,"Exploratory")
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response, Query3)

******** PROMPT *****************
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a knowledgeable medical assistant providing accurate, structured medical information.

Your responses must:
- Use medically correct terminology
- Explain symptoms, causes, diagnostics, and treatments when applicable
- Cite specific clinical evidence (drugs, dosages, procedures) where possible
- Avoid unsupported claims, absolute language ("always cures", "100%"), or speculation
- End with a brief disclaimer recommending consultation with a qualified doctor

Response format:
- **Clinical Explanation** — brief overview of the condition
- **Symptoms / Presentation** — bullet list
- **Diagnosis** — bullet list of diagnostic steps if relevant
- **Treatment Protocol** — bullet list with specific interventions
- **Disclaimer** — one-line clinical caveat

CRITICAL: Provide your answer ONCE. Do not evaluate, grade, or comment on your own response. Stop after the disclaimer.<|eot_id|><|start_he

Llama.generate: prefix-match hit


**Clinical Explanation:** Sudden patchy hair loss, also known as alopecia areata, is an autoimmune condition characterized by the loss of hair in distinct patches or circular areas. This condition can affect people of any age, but it typically starts between the ages of 15 and 25.

**Symptoms / Presentation:**

• Patchy hair loss on the scalp
• Circular or irregularly shaped bald spots
• Hair may break off close to the surface of the skin
• May occur suddenly or gradually over time

**Diagnosis:**

• Physical examination by a dermatologist or healthcare provider
• Medical history and review of symptoms
• Exclusion of other conditions that can cause hair loss, such as fungal infections, psoriasis, or thyroid disorders
• Possible biopsy to rule out skin cancer or other inflammatory conditions

**Treatment Protocol:**

• Topical corticosteroids (e.g., triamcinolone) for localized application
• Minoxidil (Rogaine) 2% solution for topical application
• Oral medications, such as finasteride 

{'word_count': 220,
 'sentence_count': 4,
 'bullet_count': 12,
 'header_count': 3,
 'paragraph_count': 8,
 'has_structure': True,
 'clinical_terms_count': 8,
 'has_disclaimer': False,
 'hedge_count': 3,
 'red_flag_count': 0,
 'looks_truncated': False,
 'query_overlap': 0.58,
 'quality_score': 0.766}

In [85]:
prompt = build_llama3_prompt(Query4)
print("******** PROMPT *****************")
print(prompt)
print("******** RESPONSE *****************")
response = llm_response(prompt,"Exploratory")
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response, Query4)

******** PROMPT *****************
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a knowledgeable medical assistant providing accurate, structured medical information.

Your responses must:
- Use medically correct terminology
- Explain symptoms, causes, diagnostics, and treatments when applicable
- Cite specific clinical evidence (drugs, dosages, procedures) where possible
- Avoid unsupported claims, absolute language ("always cures", "100%"), or speculation
- End with a brief disclaimer recommending consultation with a qualified doctor

Response format:
- **Clinical Explanation** — brief overview of the condition
- **Symptoms / Presentation** — bullet list
- **Diagnosis** — bullet list of diagnostic steps if relevant
- **Treatment Protocol** — bullet list with specific interventions
- **Disclaimer** — one-line clinical caveat

CRITICAL: Provide your answer ONCE. Do not evaluate, grade, or comment on your own response. Stop after the disclaimer.<|eot_id|><|start_he

Llama.generate: prefix-match hit


**Clinical Explanation**
A traumatic brain injury (TBI) occurs when an external force damages the brain, leading to temporary or permanent impairments in cognitive, emotional, and/or physical functioning.

**Symptoms / Presentation**

* Headache
* Confusion
* Dizziness or loss of balance
* Blurred vision
* Seizures or convulsions
* Memory impairment
* Difficulty concentrating or paying attention
* Mood changes (e.g., irritability, mood swings)
* Sleep disturbances
* Fatigue
* Slurred speech or difficulty articulating thoughts

**Diagnosis**
Diagnostic steps include:

* Physical examination and medical history
* Imaging studies: computed tomography (CT) scan, magnetic resonance imaging (MRI), and/or positron emission tomography (PET) scans to rule out other conditions
* Neuropsychological testing to assess cognitive function and behavior

**Treatment Protocol**

* **Acute Care**
	+ Rest and recovery in a controlled environment
	+ Pain management with medications such as acetaminophen or

{'word_count': 246,
 'sentence_count': 2,
 'bullet_count': 17,
 'header_count': 5,
 'paragraph_count': 8,
 'has_structure': True,
 'clinical_terms_count': 8,
 'has_disclaimer': False,
 'hedge_count': 0,
 'red_flag_count': 0,
 'looks_truncated': False,
 'query_overlap': 0.5,
 'quality_score': 0.65}

In [86]:
prompt = build_llama3_prompt(Query5)
print("******** PROMPT *****************")
print(prompt)
print("******** RESPONSE *****************")
response = llm_response(prompt,"Exploratory")
print(response)
print("********************* LLM Response Evaluation ***********************")
evaluate_response(response, Query5)

******** PROMPT *****************
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a knowledgeable medical assistant providing accurate, structured medical information.

Your responses must:
- Use medically correct terminology
- Explain symptoms, causes, diagnostics, and treatments when applicable
- Cite specific clinical evidence (drugs, dosages, procedures) where possible
- Avoid unsupported claims, absolute language ("always cures", "100%"), or speculation
- End with a brief disclaimer recommending consultation with a qualified doctor

Response format:
- **Clinical Explanation** — brief overview of the condition
- **Symptoms / Presentation** — bullet list
- **Diagnosis** — bullet list of diagnostic steps if relevant
- **Treatment Protocol** — bullet list with specific interventions
- **Disclaimer** — one-line clinical caveat

CRITICAL: Provide your answer ONCE. Do not evaluate, grade, or comment on your own response. Stop after the disclaimer.<|eot_id|><|start_he

Llama.generate: prefix-match hit


**Clinical Explanation**: A fractured leg, also known as a lower limb fracture, occurs when one or more bones in the leg are broken due to trauma, such as falling or twisting. Fractures can be classified into different types based on the severity of the break, location, and surrounding soft tissue damage.

**Symptoms / Presentation**:

• Severe pain and swelling in the affected area
• Deformity or abnormal alignment of the leg
• Limited mobility or inability to bear weight on the affected leg
• Bruising, ecchymosis, or lacerations near the fracture site
• Numbness, tingling, or loss of sensation in the lower extremities

**Diagnosis**:

• Physical examination and visual inspection of the affected area
• X-rays, computed tomography (CT), or magnetic resonance imaging (MRI) scans to confirm the diagnosis and determine the type and extent of the fracture
• Assessment of surrounding soft tissue damage and potential complications, such as compartment syndrome or nerve damage

**Treatment Pr

{'word_count': 262,
 'sentence_count': 4,
 'bullet_count': 13,
 'header_count': 3,
 'paragraph_count': 8,
 'has_structure': True,
 'clinical_terms_count': 9,
 'has_disclaimer': False,
 'hedge_count': 1,
 'red_flag_count': 0,
 'looks_truncated': False,
 'query_overlap': 0.13,
 'quality_score': 0.609}

In [72]:
import pandas as pd

# Optional per-config prompt variations
CONFIG_PROMPT_OPTIONS = {
    "Deterministic": {"constraints": None,                                       "use_cot": False},
    "Conservative":  {"constraints": "Answer in under 5 sentences.",             "use_cot": False},
    "Balanced":      {"constraints": "Answer in under 10 sentences.",            "use_cot": False},
    "Creative":      {"constraints": None,                                       "use_cot": True},
    "Exploratory":   {"constraints": None,                                       "use_cot": True},
}

QUERIES = {
    "Q1_sepsis":       Query1,
    "Q2_appendicitis": Query2,
    "Q3_alopecia":     Query3,
    "Q4_brain_injury": Query4,
    "Q5_leg_fracture": Query5,
}


def run_all() -> pd.DataFrame:
    """Run every (config, query) pair and collect per-response metrics."""
    rows = []
    for config_name, sampling in SAMPLING_PROFILES.items():
        prompt_opts = CONFIG_PROMPT_OPTIONS[config_name]
        for query_id, query_text in QUERIES.items():
            prompt   = build_llama3_prompt(query_text, **prompt_opts)
            response = llm_response(prompt, **sampling)
            metrics  = evaluate_response(response, query=query_text)

            rows.append({
                "config":   config_name,
                "query":    query_id,
                "response": response,
                **metrics,
            })
            print(f"✓ {config_name:>14} × {query_id}  →  score={metrics['quality_score']}")

    return pd.DataFrame(rows)


# Run all the profiles for each query
results_df = run_all()


# Aggregation evaluation data

# View 1: per-config summary (mean across queries)
config_summary = (
    results_df
    .groupby("config")
    .agg(
        mean_score          = ("quality_score",        "mean"),
        mean_word_count     = ("word_count",           "mean"),
        mean_clinical_terms = ("clinical_terms_count", "mean"),
        mean_query_overlap  = ("query_overlap",        "mean"),
        pct_disclaimer      = ("has_disclaimer",       "mean"),
        pct_truncated       = ("looks_truncated",      "mean"),
        mean_hedges         = ("hedge_count",          "mean"),
        mean_red_flags      = ("red_flag_count",       "mean"),
    )
    .round(2)
    .reindex(SAMPLING_PROFILES.keys())   # preserve sweep order
)

print("\n=== Config Summary (mean across 5 queries) ===")
print(config_summary.to_markdown())


# View 2: full quality_score matrix (config × query)
score_matrix = (
    results_df
    .pivot(index="config", columns="query", values="quality_score")
    .reindex(SAMPLING_PROFILES.keys())
    .reindex(columns=QUERIES.keys())
)
score_matrix["mean"] = score_matrix.mean(axis=1).round(2)

print("\n=== Quality Score Matrix ===")
print(score_matrix.to_markdown())


# View 3: per-query best config
best_per_query = (
    results_df
    .loc[results_df.groupby("query")["quality_score"].idxmax()]
    [["query", "config", "quality_score"]]
    .reset_index(drop=True)
)

print("\n=== Best Config per Query ===")
print(best_per_query.to_markdown(index=False))

Llama.generate: prefix-match hit


✓  Deterministic × Q1_sepsis  →  score=0.636


Llama.generate: prefix-match hit


✓  Deterministic × Q2_appendicitis  →  score=0.673


Llama.generate: prefix-match hit


...

✓    Exploratory × Q3_alopecia  →  score=0.669


Llama.generate: prefix-match hit


✓    Exploratory × Q4_brain_injury  →  score=0.608


Llama.generate: prefix-match hit


✓    Exploratory × Q5_leg_fracture  →  score=0.616

=== Config Summary (mean across 5 queries) ===
| config        |   mean_score |   mean_word_count |   mean_clinical_terms |   mean_query_overlap |   pct_disclaimer |   pct_truncated |   mean_hedges |   mean_red_flags |
|:--------------|-------------:|------------------:|----------------------:|---------------------:|-----------------:|----------------:|--------------:|-----------------:|
| Deterministic |         0.68 |             257   |                   8.4 |                 0.5  |                0 |               0 |           1   |                0 |
| Conservative  |         0.64 |             138.4 |                   7.2 |                 0.39 |                0 |               0 |           0.6 |                0 |
| Balanced      |         0.64 |             145.4 |                   7.8 |                 0.38 |                0 |               0 |           0.4 |                0 |
| Creative      |         0.66 |         